# Cấu hình

In [1]:
REPO_URL = "https://github.com/thieuphong220203/vietnamese-voice-qa.git"
BRANCH = "main"
PROJECT_DIR = "/content/AdvancedML"

# Không cache model trên Google Drive vì file model đã bị lưu sai thành 79 bytes.
USE_DRIVE_CACHE = False

# Audio dùng để test ASR.
SAMPLE_AUDIO = "data/samples/fleurs_vi/audio/sample_000.wav"

# Mount Drive và cấu hình model cache

In [2]:
import os

if USE_DRIVE_CACHE:
    raise RuntimeError(
        "Không nên dùng Drive cache cho PhoWhisper vì file trọng số "
        "có thể bị lưu thành file con trỏ 79 bytes."
    )

os.environ["HF_HOME"] = "/content/huggingface"
os.environ["HF_HUB_CACHE"] = "/content/huggingface/hub"
os.environ["HF_HUB_DISABLE_XET"] = "1"

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_HUB_CACHE:", os.environ["HF_HUB_CACHE"])

HF_HOME: /content/huggingface
HF_HUB_CACHE: /content/huggingface/hub


# Update code

In [3]:
import os
import subprocess
from pathlib import Path

project_path = Path(PROJECT_DIR)
git_path = project_path / ".git"


def git(*args: str) -> None:
    subprocess.run(
        ["git", "-C", str(project_path), *args],
        check=True,
    )


if git_path.is_dir():
    print("Repository đã tồn tại, đang cập nhật...")

    # Repo cũ có thể đã được clone với --single-branch.
    # Cho phép origin theo dõi mọi nhánh từ giờ trở đi.
    git("remote", "set-branches", "origin", "*")
    git("fetch", "origin")

elif project_path.exists() and any(project_path.iterdir()):
    raise RuntimeError(
        f"{project_path} đã tồn tại nhưng không phải Git repository. "
        "Hãy đổi PROJECT_DIR hoặc kiểm tra thư mục này."
    )

else:
    print("Đang clone repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(project_path)],
        check=True,
    )

# Báo lỗi rõ nếu tên nhánh không tồn tại trên origin.
remote_ref = f"refs/remotes/origin/{BRANCH}"
result = subprocess.run(
    ["git", "-C", str(project_path),
     "show-ref", "--verify", "--quiet", remote_ref],
)
if result.returncode != 0:
    raise RuntimeError(f"Không tìm thấy nhánh origin/{BRANCH}")

local_branch = subprocess.run(
    ["git", "-C", str(project_path),
     "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"],
)

if local_branch.returncode == 0:
    git("switch", BRANCH)
else:
    git("switch", "--track", f"origin/{BRANCH}")

git("pull", "--ff-only", "origin", BRANCH)

os.chdir(project_path)

actual_branch = subprocess.check_output(
    ["git", "-C", str(project_path), "branch", "--show-current"],
    text=True,
).strip()
commit = subprocess.check_output(
    ["git", "-C", str(project_path), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print(f"Project: {project_path}")
print(f"Branch:  {actual_branch}")
print(f"Commit:  {commit}")

Đang clone repository...
Project: /content/AdvancedML
Branch:  main
Commit:  adcc184


In [4]:
print("Nhánh thực tế:", subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())

Nhánh thực tế: main


In [5]:
# !git -C /content/AdvancedML remote set-branches --add origin feature/qa-qwen-streaming
# !git -C /content/AdvancedML fetch origin
# !git -C /content/AdvancedML switch --track origin/feature/qa-qwen-streaming
# !git -C /content/AdvancedML status --short --branch

# Cài Dependecy

In [6]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dtln.txt",
    ],
    check=True,
)

print("Dependencies đã sẵn sàng.")

Dependencies đã sẵn sàng.


In [ ]:
# !apt-get update -qq && apt-get install -y ffmpeg

^C


In [7]:
import shutil
print("FFmpeg:", shutil.which("ffmpeg"))

FFmpeg: /usr/bin/ffmpeg


# Kiểm tra GPU và thư viện

In [8]:
import gradio
import torch
import transformers

print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Gradio:       {gradio.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
else:
    print(
        "Cảnh báo: app đang dùng CPU. "
        "Hãy chọn Runtime → Change runtime type → GPU."
    )

PyTorch:      2.14.1+cu130
Transformers: 5.22.1
Gradio:       6.31.0
CUDA:         True
GPU:          NVIDIA L4


# Nạp checkpoint fine-tuned

In [9]:
from google.colab import drive
import os
import shutil
from pathlib import Path

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Đặt đường dẫn checkpoint trên Drive và trong project Colab
DRIVE_BEST_PT = Path("/content/drive/MyDrive/vvqa/checkpoints/separation/finetuned/best.pt")
TARGET_DIR = Path("/content/AdvancedML/checkpoints/separation/finetuned")
TARGET_FILE = TARGET_DIR / "best.pt"

TARGET_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_BEST_PT.exists():
    shutil.copy(DRIVE_BEST_PT, TARGET_FILE)
    print(f"Đã copy best.pt thành công! Dung lượng: {TARGET_FILE.stat().st_size / 1e6:.1f} MB")
else:
    print(f"❌ Không tìm thấy: {DRIVE_BEST_PT}")
    print("Vui lòng kiểm tra lại đường dẫn file trên Google Drive của bạn.")

Mounted at /content/drive
✅ Đã copy best.pt thành công! Dung lượng: 113.1 MB


In [ ]:
# Copy checkpoint DTLN fine-tuned từ Drive
DRIVE_DTLN_H5 = Path("/content/drive/MyDrive/vvqa/checkpoints/enhancement/dtln/DTLN_vi_snr5_lr1e-4_run01.weights.h5")
TARGET_DTLN_DIR = Path("/content/AdvancedML/checkpoints/enhancement/dtln")
TARGET_DTLN_FILE = TARGET_DTLN_DIR / "DTLN_vi_snr5_lr1e-4_run01.weights.h5"

TARGET_DTLN_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_DTLN_H5.exists():
    shutil.copy(DRIVE_DTLN_H5, TARGET_DTLN_FILE)
    print(f"Đã copy DTLN checkpoint thành công! ({TARGET_DTLN_FILE.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Chưa có {DRIVE_DTLN_H5} trên Drive. DTLN sẽ tự động dùng pretrained model gốc.")

In [ ]:
# Trong configs/pipeline.yaml hoặc đặt biến môi trường:
import os
# Giữ GPU cho PyTorch, không để TensorFlow cấp phát độc quyền
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# Chạy app

In [10]:
%cd /content/AdvancedML
!python -u -c "print('Python chạy được'); import app.main; print('Import app.main thành công')"

/content
Python chạy được
Import app.main thành công


In [ ]:
%cd /content/AdvancedML
!python -u -m src.tts.infer --text "Thủ đô của Hàn Quốc là Seoul."

/content
Loading TTS from configs/tts.yaml...
Model load: 9.400 s
Audio: /content/AdvancedML/outputs/audio/tts_sample.wav
Duration: 1.760 s
First audio: 0.501 s
Generation: 2.045 s
RTF: 1.162


In [11]:
%cd /content/AdvancedML
!python -u app/main.py --config configs/pipeline.yaml --host 0.0.0.0 --port 7860 --share

/content
Đang load pipeline từ: /content/AdvancedML/configs/pipeline.yaml
hyperparams.yaml: 100% 1.52k/1.52k [00:00<00:00, 5.05MB/s]
masknet.ckpt: 100% 113M/113M [00:01<00:00, 72.2MB/s]
encoder.ckpt: 100% 17.3k/17.3k [00:00<00:00, 46.3kB/s]
decoder.ckpt: 100% 17.3k/17.3k [00:00<00:00, 45.6kB/s]
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
Loading model: vinai/PhoWhisper-small
Device: cuda
Dtype: torch.float16
preprocessor_config.json: 100% 339/339 [00:00<00:00, 2.16MB/s]
config.json: 100% 1.33k/1.33k [00:00<00:00, 5.16MB/s]
tokenizer_config.json: 100% 805/805 [00:00<00:00, 4.97MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 85.5MB/s]
tokenizer.json: 100% 2.20M/2.20M [00:00<00:00, 145MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 98.9MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 20.0MB/s]
added_tokens.json: 100% 2.08k/2.08k [00:00<00:00, 8.12MB/s]
special_tokens_map.json: 100% 2.08k/2.08k [00:00<00:00, 8.89M

In [ ]:
# import torch
# from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# # Ép torch.load tương thích với PyTorch trên Colab
# _orig_load = torch.load
# torch.load = lambda *args, **kwargs: _orig_load(*args, **{**kwargs, "weights_only": False})

# model_id = "vinai/PhoWhisper-small"

# print("1. Loading processor...")
# processor = AutoProcessor.from_pretrained(model_id)
# print("Processor OK")

# print("\n2. Downloading and loading model (chờ tải ~900MB)...")
# model = AutoModelForSpeechSeq2Seq.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16,
#     force_download=True,  # 👈 Bắt buộc tải lại bản sạch hoàn chỉnh
# )
# print("Model OK!")
